# Sense cleanup test: does an LLM pre-pass over raw dictionary senses help?

Quick gut-check before building real caching machinery for the
"LLM cleans up messy dictionary senses" idea discussed for phrase decomposition
(see memory: `phrase_decomposition_design.md`).

The real problem: raw Datuk entries for a headword are a dense, unlabeled list
of senses (archaic phrasing, no usage notes, no example sentences) — unlike an
idealized modern dictionary (e.g. a "bank" entry with clearly labeled
"financial institution" vs "river bank" senses, each with example usage).

**Question**: if an LLM rewrites the raw senses into a short labeled list
first (sense + modern gloss + usage label), does the *final* answer-LLM do a
better job picking the right sense — compared to being handed the raw text
directly?

Real example under test: `"Dayavayi Irikkoo sahodara"` (Manglish for
"please sit, brother") — the word ഇരിക്കുക (`Irikkoo`) has to resolve to
"to sit", not one of its many other senses (to live, to behave, to occur,
to observe duties, to fall...).

In [1]:
import hydra
from omegaconf import DictConfig

from linguaalayam.database import build_engine, build_session_factory
from linguaalayam.embeddings import EmbeddingService
from linguaalayam.env import load_env
from linguaalayam.llm.adapters.togetherai import TogetherAIAdapter
from linguaalayam.rag.tools import DictionaryTools

load_env()

with hydra.initialize_config_dir(config_dir=__import__("os").path.abspath("../config"), version_base=None):
    cfg: DictConfig = hydra.compose(config_name="config")

service = EmbeddingService(cfg.embedding)
engine = build_engine(cfg.database)
session_factory = build_session_factory(engine)
tools = DictionaryTools(session_factory, service)
llm = TogetherAIAdapter(model="Qwen/Qwen3.5-9B", temperature=0.0, max_tokens=512)

/home/sachin_nandakumar/projects/LinguAalayam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Pull the real raw entries for ഇരിക്കുക

In [2]:
headword = "ഇരിക്കുക"
raw_entries = tools.exact_lookup(headword) or tools.fuzzy_lookup(headword, top_k=5)
raw_text = "\n".join(f"- {e['embed_text']}" for e in raw_entries)
print(raw_text)

- word: ഇരിക്കുക
  [general] പാർക്കുക; ജീവിക്കുക
- word: ഇരിക്കുക
  [ക്രി.] ഒരു അനുപ്രയോഗം; ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക; പൃഷ്ഠം ഉറപ്പിച്ചു സ്ഥിതിചയ്യുക, ആസനസ്ഥനാവുക; തോന്നുക; അനുഷ്ഠിക്കുക, ആചരിക്കുക; ഏർപ്പെടുക; ഉരിക്കുക; താഴുക


## 2. LLM cleanup pass — rewrite raw senses as a labeled list

In [3]:
CLEANUP_SYSTEM = """You are cleaning up a raw, messy Malayalam dictionary entry so a \
future reader can quickly pick the right sense. You will be given a headword \
and a list of raw definition strings (often archaic, dense, or repetitive). \
Rewrite them as a short numbered list of distinct senses. For each sense give:
- a one-line modern English gloss
- a short label for when/how it's used (e.g. "most common, everyday", "formal/literary", "idiomatic")
Drop near-duplicate senses. Keep it under 6 senses. Be concise."""

cleaned = llm.complete(CLEANUP_SYSTEM, f"Headword: {headword}\n\nRaw definitions:\n{raw_text}")
print(cleaned)

1. **To sit / to be seated**
   *Most common, everyday*

2. **To live / to exist / to be in a state**
   *General usage (e.g., "he is living," "it is raining")*

3. **To behave / to conduct oneself**
   *Formal or descriptive (e.g., "how he is behaving")*

4. **To occur / to seem / to appear**
   *Idiomatic or abstract (e.g., "it seems," "it occurred to me")*

5. **To perform / to observe (a ritual or duty)**
   *Formal/literary (e.g., "observing a fast")*

6. **To fall / to descend**
   *Archaic or poetic (e.g., "the sun fell")*


## 3. A/B test: final answer using RAW senses vs CLEANED senses

Same final-synthesis prompt shape used in `_decompose_and_lookup` /
`_DECOMPOSITION_SYNTHESIS_SYSTEM` (web.py), simplified to a single word so the
cleanup variable is isolated.

In [4]:
phrase = "Dayavayi Irikkoo sahodara"

final_system_raw = (
    "You are a Malayalam dictionary assistant. A user asked about this Manglish "
    "phrase. Below are raw dictionary entries for each word (possibly messy/archaic). "
    "Use your own knowledge of Malayalam too. Explain what the phrase means, "
    "and for each word say which sense applies."
)
final_user_raw = f"Phrase: {phrase}\n\nWord: Irikkoo (ഇരിക്കുക)\nRaw senses:\n{raw_text}"

final_system_cleaned = final_system_raw.replace(
    "Below are raw dictionary entries", "Below are cleaned-up dictionary entries"
)
final_user_cleaned = f"Phrase: {phrase}\n\nWord: Irikkoo (ഇരിക്കുക)\nCleaned senses:\n{cleaned}"

answer_raw = llm.complete(final_system_raw, final_user_raw)
answer_cleaned = llm.complete(final_system_cleaned, final_user_cleaned)

print("=== FINAL ANSWER using RAW senses ===")
print(answer_raw)

=== FINAL ANSWER using RAW senses ===
Here is the explanation for the phrase **"Dayavayi Irikkoo sahodara"** (ദയവായി ഇരിക്കൂ സഹോദര).

### **Phrase Meaning**
**"Please sit, brother."**

This is a polite and affectionate request used when asking someone (specifically a male friend, relative, or someone of equal status) to take a seat. The use of "Sahodara" (brother) adds a tone of closeness or respect.

---

### **Word Analysis: Irikkoo (ഇരിക്കൂ)**

**Word:** ഇരിക്കുക (Irikkuka)
**Form used:** ഇരിക്കൂ (Irikkoo) - This is the **Imperative Mood** (command/request) of the verb.

**Which Sense Applies?**
Based on the context of the phrase ("Please sit"), the applicable sense from your raw data is:
> **ആസനസ്ഥനാവുക** (To become seated / To sit down)

*Note: While the raw data lists many meanings like "to live" (ജീവിക്കുക) or "to exist" (ഉണ്ടായിരിക്കുക), in this specific sentence, the meaning is strictly physical sitting.*

**Detailed Breakdown:**
1.  **Irikkuka (ഇരിക്കുക):** The base verb mean

In [5]:
print("=== FINAL ANSWER using CLEANED senses ===")
print(answer_cleaned)

=== FINAL ANSWER using CLEANED senses ===
The phrase **"Dayavayi Irikkoo sahodara"** translates to:

> **"Please sit, brother."**
> *(Or more naturally: "Brother, please take a seat.")*

Here is the breakdown of the phrase and the specific sense of the word **Irikkoo** used here:

### Phrase Analysis
*   **Dayavayi (ദയവായി):** Please / Kindly.
*   **Irikkoo (ഇരിക്കൂ):** Sit! (Imperative form).
*   **Sahodara (സഹോദര):** Brother.

### Word Breakdown: Irikkoo (ഇരിക്കുക)
In this specific context, the sense applied is:

**1. To sit / to be seated**
*   **Why:** The speaker is giving a direct instruction to the listener ("brother") to take a physical seat. This is the most common, everyday usage of the verb.
*   **Context:** It is used in social situations where someone is invited to rest or join a conversation by sitting down.

*(Note: While "Irikkoo" can also mean "to live" or "to exist" in other contexts, such as "Ninnu irikkunnathu" (He is living), that meaning does not fit the command s

## Result (first real run)

Both the RAW-senses prompt and the CLEANED-senses prompt correctly resolved
`Irikkoo` to "to sit" and explicitly reasoned about *why* the other senses
(to live, to behave, to occur) didn't fit the context.

This contradicts the earlier combinatorial-logprob-scoring experiments
(`phrase_decomposition_sense_ranking.ipynb`, section 5), which failed to rank
the correct sense first — but those scored *context-free label strings* in
isolation, never giving the LLM the full phrase + its own reasoning in one
pass. This test instead gives the LLM the full sentence context and lets it
reason in natural language, which the earlier approach deliberately avoided
(it was testing *mechanical* scoring, not LLM reasoning).

**Caveat**: single phrase, single word, single run — not yet a systematic
comparison. The real open question is whether RAW is already good enough
most of the time (making the cleanup pass pure cost with no benefit) or
whether cleanup pays off on harder/more cluttered headwords than ഇരിക്കുക.
Needs a wider sample before deciding whether options #2/#3 (LLM-cleanup,
cached or not) are worth building at all — it's possible the real
synthesis prompt (which already works this way in production) is already
sufficient without any new dictionary-cleanup machinery.

## 4. Grounding: force the LLM to cite the exact dictionary fragment it used

Both A/B answers above picked "to sit" correctly, but neither was required to
point at *which specific substring of the real dictionary entry* backs that
choice — the LLM could just as easily have answered from its own general
Malayalam knowledge and ignored the DB text entirely. That's not acceptable:
the dictionary is the knowledge base, the LLM's job is to pick the right
entry from it and make it understandable — not to freelance.

This variant adds an explicit instruction: quote the exact matching
definition fragment from the raw dictionary text first, *then* explain it in
plain language. If the model can't quote a real substring, that's a sign the
prompt (or the dictionary data) isn't actually grounding the answer.

In [6]:
GROUNDED_SYSTEM = (
    "You are a Malayalam dictionary assistant. A user asked about a Manglish "
    "phrase. Below are the RAW dictionary entries for one word in that phrase "
    "(exact text from the knowledge base, not paraphrased). "
    "First, quote the exact substring from the raw entries below that is the "
    "correct sense for this context — copy it verbatim, do not paraphrase it "
    "in this step. Then, in a second step, explain that quoted sense in plain "
    "modern English so a learner understands it. Do not invent a definition "
    "that isn\'t present in the raw text — if nothing fits well, say so "
    "explicitly instead of guessing from general knowledge."
)
grounded_user = f"Phrase: {phrase}\n\nWord: Irikkoo (ഇരിക്കുക)\nRaw dictionary entries:\n{raw_text}"

answer_grounded = llm.complete(GROUNDED_SYSTEM, grounded_user)
print(answer_grounded)

**Exact substring from raw entries:**
"ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക"

**Explanation in plain modern English:**
In this context, the word "Irikkoo" (ഇരിക്കുക) means "to live," "to behave," "to be situated," or "to exist." It describes the state of being alive or the way someone is living their life.


In [7]:
# Verify the quoted fragment is actually a real substring from the DB text,
# not a paraphrase/hallucination.
import re

quoted = re.findall(r"[ഀ-ൿ,\s]{3,}", answer_grounded)
hits = [q.strip() for q in quoted if q.strip() and q.strip() in raw_text]
print("Malayalam substrings in the answer that are verbatim present in raw_text:")
for h in hits:
    print(" -", repr(h))
print("\nAny verbatim hit found:", bool(hits))

Malayalam substrings in the answer that are verbatim present in raw_text:
 - 'ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക'
 - 'ഇരിക്കുക'

Any verbatim hit found: True


## Updated conclusion — negative result, important

Forcing verbatim-quote-then-explain **got the wrong sense**: the model quoted
`"പാർക്കുക; ജീവിക്കുക"` ("to live / to reside") and built a wrong explanation
("Live kindly, brother") around it — even though the correct substring
`ആസനസ്ഥനാവുക` ("to become seated") is sitting right there in the same raw
entry it was given.

This is the opposite of what section 3 showed. The difference: sections 3's
looser prompt ("use the entries as context, explain the phrase") let the
model reason about the *whole sentence* first and arrive at "to sit"
correctly both times. Section 4's strict prompt ("quote the exact matching
substring first") made the model grab the *first plausible-looking*
fragment early in the list and commit to it before reasoning about whether
it actually fits the sentence — raw entries here are ordered unhelpfully
(the "to live" sense comes before "to sit" in the DB text), and the
quote-first instruction seems to encourage picking by position/salience
rather than by fit.

**Conclusion so far**: "ground it in a real DB fragment" and "let the LLM
reason freely over the DB context" are in tension, not additive, with the
current unordered/unlabeled raw dictionary data and a single-shot prompt.
Grounding is desirable (the user explicitly wants answers to come from the
actual knowledge base, not the LLM's freelance general knowledge) — but this
naive version of grounding regressed accuracy on this exact example.

**Next thing worth trying** (not yet done): ask the LLM to reason about the
sentence context *first*, state in plain language which sense it thinks
applies, and only *then* go back and point to/quote the specific raw
fragment that supports that conclusion — reversing the order from this
cell's prompt (reason → ground, not ground → reason). That keeps the
grounding requirement (every answer must cite real DB text, not freelance
knowledge) while avoiding the premature-commitment failure seen here. Not
yet tested against a second query sample, so this is a hypothesis, not a
proven fix.

## 5. Reverse the order: reason about context first, then cite the DB fragment

Section 4's prompt asked for the quote *before* reasoning and got the wrong
sense. This section tests the reverse order: explain what the word means in
context first, decide on a sense in plain English, *then* go back and point
to the specific raw dictionary fragment that supports that conclusion — same
grounding requirement, different order.

In [8]:
REASON_THEN_GROUND_SYSTEM = (
    "You are a Malayalam dictionary assistant. A user asked about a Manglish "
    "phrase. Below are the RAW dictionary entries for one word in that phrase "
    "(exact text from the knowledge base). "
    "First, reason about the word\'s role in the full phrase and decide, in "
    "plain English, what it means here. Then, as a second and final step, go "
    "back to the raw entries and quote verbatim the exact substring that "
    "supports your conclusion. If no substring in the raw entries actually "
    "supports your conclusion, say so explicitly instead of forcing a match."
)
rtg_user = f"Phrase: {phrase}\n\nWord: Irikkoo (ഇരിക്കുക)\nRaw dictionary entries:\n{raw_text}"

answer_reason_then_ground = llm.complete(REASON_THEN_GROUND_SYSTEM, rtg_user)
print(answer_reason_then_ground)

In the phrase "Dayavayi Irikkoo sahodara" (meaning "Be kind, brother" or "Have mercy, brother"), the word **Irikkoo** functions as a polite imperative verb. Here, it does not mean "to sit" or "to live" in a literal sense. Instead, it is used in the sense of "to be," "to exist," or "to remain," effectively translating to "be" or "stay" in this context of urging someone to maintain a certain state of kindness.

Looking at the provided raw dictionary entries, the substring that supports this conclusion is:
**"ഉണ്ടായിരിക്കുക"**

This entry appears in the second definition block: `[ക്രി.] ഒരു അനുപ്രയോഗം; ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക; ...`. The term "ഉണ്ടായിരിക്കുക" (to be/exist) aligns with the imperative usage of "be kind" in the phrase.


In [9]:
quoted = re.findall(r"[ഀ-ൿ,\s]{3,}", answer_reason_then_ground)
hits = [q.strip() for q in quoted if q.strip() and q.strip() in raw_text]
print("Verbatim DB substrings cited:", [repr(h) for h in hits])
print("Correctly grounded in the 'to sit' fragment specifically:",
      any("ആസനസ്ഥനാവുക" in h for h in hits))

Verbatim DB substrings cited: ["'ഉണ്ടായിരിക്കുക'", "'ക്രി'", "'ഒരു അനുപ്രയോഗം'", "'ജീവിക്കുക, വർത്തിക്കുക, സ്ഥിതിചെയ്യുക, ഉണ്ടായിരിക്കുക'", "'ഉണ്ടായിരിക്കുക'"]
Correctly grounded in the 'to sit' fragment specifically: False


## Final conclusion for this test session — mixed/negative, three runs

Reversing the order did not fix it either, and failed in a *new* way: this
run mistranslated **"Dayavayi"** itself as "be kind" instead of "please",
which cascaded into the model reasoning that "Irikkoo" must mean "continue
to be [kind]" — then, to its credit, honestly refused to ground that
conclusion in any raw DB fragment, since none actually supports it. So the
refusal-to-force-a-match behavior worked exactly as instructed; the root
failure was an unrelated translation error on a *different* word in the
phrase, not a grounding-order problem at all.

**Across all three grounding attempts (sections 3, 4, 5) the result was
inconsistent run to run** — correct in section 3 (twice, loose prompt),
wrong in section 4 (strict quote-first), wrong-for-a-different-reason in
section 5 (reason-then-ground). This is consistent with the earlier
`phrase_decomposition_sense_ranking.ipynb` findings: single-shot LLM
judgment on this exact phrase is not reliably stable, and no prompt
ordering tried so far (here or there) fixes it consistently.

**Honest takeaway for the user's request** ("refer as much as possible from
the DB, let the LLM make it explainable"): that's the right design goal, but
this test session shows a single-shot "pick + quote + explain" prompt isn't
reliable enough to deliver it today — forcing a quote can force a wrong
quote. A real fix likely needs more than a prompt tweak: either (a) multiple
samples/self-consistency voting across repeated calls, (b) structured
output forcing the model to pick one specific entry *index* from a numbered
list (constrained choice) rather than free-text quoting, or (c) the
sense-cleanup-and-cache idea (options #2/#3) done well enough that the
cached version is unambiguous enough not to need this fragile quoting step
at query time at all. None of these have been tested yet — this notebook
only establishes that naive grounding, in either order, is not yet solid.

## 6. Broader sample: three real sentences

Sections 3-5 only tested one word in one phrase. This section runs the same
"ground every word in a real DB entry, then explain" idea across three
genuinely different real sentences:

1. `"enthaado thanikk vende?"` — Manglish, colloquial question ("what do you
   need/want?")
2. A full native-script Malayalam sentence (encyclopedic, about a lake in
   Azerbaijan) — no Manglish/transliteration involved, tests whether
   grounding works on already-correct-script, lower-ambiguity input.
3. A full native-script Malayalam sentence (mythology — Mahabali's wife) —
   tests a domain (Hindu mythology/proper nouns) the dictionary almost
   certainly has little or no coverage for, which is itself a useful signal:
   does the model correctly say "not in the dictionary" rather than forcing
   a fake grounding?

In [10]:
from linguaalayam.transliteration.varnam import manglish_to_malayalam

TEST_SENTENCES = [
    "enthaado thanikk vende?",
    "അസർബൈജാനിലെ കുർ-അറാസ് താഴ്ന്ന പ്രദേശങ്ങളിൽ സ്ഥിതിചെയ്യുന്ന ഒരു വലിയ ലവണജല തടാകമാണ് അഗ്ഗോൾ തടാകം",
    "ഹിന്ദു പുരാണങ്ങളിലെ അസുര രാജാവായ മഹാബലിയുടെ ഭാര്യയാണ് വിന്ധ്യാവലി",
]


def decompose(sentence: str) -> list[tuple[str, str]]:
    """For each word: try exact/fuzzy lookup directly; if no hit and the word
    is Latin-script, try Varnam Manglish candidates. Returns (word, raw_text)
    pairs, skipping words with no dictionary hit at all (mirrors the real
    `_decompose_and_lookup` behaviour in web.py on the feature branch)."""
    results = []
    for word in sentence.strip("?.,!").split():
        entries = tools.exact_lookup(word) or tools.fuzzy_lookup(word, top_k=3)
        if not entries and word.isascii():
            for candidate in manglish_to_malayalam(word):
                entries = tools.exact_lookup(candidate) or tools.fuzzy_lookup(candidate, top_k=3)
                if entries:
                    break
        if entries:
            raw = "\n".join(f"- {e['embed_text']}" for e in entries)
            results.append((word, raw))
    return results


for s in TEST_SENTENCES:
    print("=" * 70)
    print("SENTENCE:", s)
    for word, raw in decompose(s):
        print(f"\n  [{word}]")
        print("  " + raw.replace("\n", "\n  "))

SENTENCE: enthaado thanikk vende?



  [enthaado]
  - word: എന്ത്
    [സ.നാ.] എത്ര, എത്രമാത്രം; വ്യാക്ഷേപകസർവനാമമായും പ്രയോഗം. ഉദാ: എന്തുപറയുന്നുവോ അങ്ങനെതന്നെ; ഏതുകാരണവശാൽ, എന്തുകൊണ്ട്; നാമവിശേഷണമായും പ്രയോഗം. (പ്ര.) ആരെന്തു പറഞ്ഞാലെന്ത്? എന്തു ചെയ്താലും അതു നടത്തണം. എന്തെന്നില്ലാതെ = പറഞ്ഞറിയിക്കാൻ നിവൃത്തിയില്ലാത്തവണ്ണം വളരെയധികം; "ചോദ്യാർഥം, ഒന്ന് ഇന്നതെന്നോ ഇന്നതരത്തിൽ ഉള്ളതെന്നോ അറിയാൻവേണ്ടി ചോദിക്കുമ്പോൾ പ്രയോഗിക്കുന്ന സർവനാമശബ്ദം. ""എ"" എന്ന ചുട്ടെഴുത്തിനോട് അനുനാസികംചേർന്ന നപുംസകരൂപം. എ-ന്-ത് = എന്ത്"; വ്യാ. ആലോചന, അത്ഭുതം മുതലായ അർത്ഥങ്ങൾ ദ്യോതിപ്പിക്കുന്നത്

  [thanikk]
  - word: തനിക്ക്
    [നാ.] "അവനവൻ (പ്ര.) തനിക്കുതാൻപോന്ന = സ്വന്തംകാര്യം നോക്കാൻ പ്രാപ്തിയുള്ള. ""തനിക്കുതകാത്തകലം ഉടഞ്ഞാലെന്ത് ഇരുന്നാലെന്ത്? തനിക്കുതാനും പുരയ്ക്കുതൂണും"" (പഴ.)"; താൻ എന്ന സർവനാമത്തിൻറെ ഉദ്ദേശികാവിഭക്തിരൂപം

  [vende]
  - word: വേണ്ട2
    ["{""വി. ഭൂ.പേ.""}"] വേണ്ടുന്ന, ആവശ്യമുള്ള. ഉദാഃ വേണ്ടപ്പെട്ടവൻ
  - word: വേണ്ടി
    [അവ്യ.] ആവശ്യമായിട്ട്, വേണ്ടീട്ട്. ഉദാഃ പോകാൻവേണ്ടി
  - word: വേണ്ടർ
    [നാ.] വേണ്ടാത്തവർ, ശത്രുക്കൾ
SEN


  [താഴ്ന്ന]
  - word: താഴുക
    [ക്രി.] ചായുക; യോഗ്യതകുറയുക, മോശമാകുക; കുറയുക; മുങ്ങുക (ജലത്തിലെന്നപോലെ); കീഴ്പ്പെടുക, തോൽക്കുക; കുനിയുക (ആദരസൂചകമായോ അപമാനംകൊണ്ടോ മറ്റോ); താഴെയാകുക (മുകളിൽനിന്നു ക്രമമായി കീഴോട്ടുപോകുക); ആഴം കൂടുക, കുഴിയുക (സമീപപ്രദേശത്തെക്കാൾ ഉയരം കുറയുക)

  [സ്ഥിതിചെയ്യുന്ന]
  - word: സ്ഥിതി
    [general] ഇരിപ്പു്; നിശ്ചയം; നില്പു്; ഒരു ദിക്കിൽതന്നെ സ്ഥിരമായി ചെയ്യുന്ന ഇരിപ്പു്
  - word: സ്ഥിതി
    [നാ.] നിലം; ഗ്രഹണകാലം; സ്ഥിരത; വിരാമം; പദവി; ആദരം; സദാചാരനിഷ്ഠ; ഉണ്ടായിരിക്കുന്ന അവസ്ഥ; നിലവിലുള്ള അവസ്ഥ; അവസ്ഥ; ഭവനം; ക്ഷേമം; അതിർത്തി; നിശ്ചയം; രക്ഷണം; മാംസം

  [ഒരു]
  - word: ഒരു
    [വി.] ഏകദേശം, ഏതാണ്ട്; പേരച്ചപ്രത്യയമായും നിരർഥകമായും പ്രയോഗം. (പ്ര.) പ്രുകണക്കിൽ, പോയോരു; അനിർദിഷ്ടവാചിയായും പ്രയോഗം, എന്തൊരു ഭംഗി ഇത്യാദി; പലതല്ലാത്ത, ഏകമായ, ഒറ്റയായ; അവധാരകാർഥത്തിലും പ്രയോഗം, വ്യത്യസ്തമല്ലാത്ത, അന്യമല്ലാത്ത, ഒരേ, ഒറ്റ
  - word: ഒരു
    [adj.] ഒന്നു്

  [വലിയ]
  - word: വലിയ
    [വി.] സമ്പത്തും കീർത്തിയുമുള്ള; പുഷ്ടിയുള്ള; ശക്തിയുള്ള; വലുതായ, ചെറുതല്ലാത്ത
  - word: വലിയ



  [ഹിന്ദു]
  - word: ഹിന്ദു
    [നാ.] ഹിന്ദുധർമം അനുഷ്ഠിക്കുന്നവൻ; ഭാരതത്തിലെ പൗരാണികമതം, ഹിന്ദുമതം

  [പുരാണങ്ങളിലെ]
  - word: പുരാണം
    [നാ.] പുരാവൃത്തം, പഴങ്കഥ; സർഗം പ്രതിസർഗം വംശം മന്വന്തരം വംശാനുചരിതം എന്നിവയഞ്ചും ഉൾപ്പെടുന്ന ഗ്രന്ഥം (ബ്രഹ്മാവിഷ്ണുമഹേശന്മാരുടെ അവതാരാദിലീലകൾ ഉൾപ്പെടുന്നതും വേദങ്ങളിലും ഉപനിഷത്തുകളിലും തന്ത്രങ്ങളിലുമുള്ള തത്ത്വങ്ങൾ വ്യാഖ്യാനിക്കുന്ന കഥകളോ കഥാഭാഗങ്ങളോ ഉള്ളവയും പ്രപഞ്ചരഹസ്യങ്ങളെയും ധർമനീതിയെയും സോദാഹരണം വിവരിക്കുന്നവയുമായ ഗ്രന്ഥങ്ങൾക്ക് പൊതുവെ പറയുന്ന പേര്. പുരാണങ്ങൾ അനേകം ഉള്ളവയിൽ പ്രധാനപ്പെട്ടവ പതിനെട്ട്); പണ്ടത്തേത്, വളരെ പഴയത്; ഒരു നാണയം 80 കവടി മൂല്യമുള്ളത്

  [അസുര]
  - word: അസുര
    [വി.] സേ്താത്രരഹിതമായ

  [രാജാവായ]
  - word: രാജാവ്
    [നാ.] രാജ്യഭരണം നിർവഹിക്കുന്ന ആൾ, നാടുവാഴി; ക്ഷത്രിയൻ; ചന്ദ്രൻ; കർപ്പൂരം; ഇന്ദ്രൻ; യക്ഷൻ; യുധിഷ്ഠിരൻ

  [മഹാബലിയുടെ]
  - word: മഹാബലി
    [general] ഒരസുരരാജാവു്; ചക്രവർത്തി
    notes: പ്രഹ്ലാദപുത്രനായ വിരോചനന്റെ പുത്രനായി ജനിച്ച ഒരു ദൈത്യരാജാവായിരുന്നു മഹാബലി.
  ഇദ്ദേഹം തപസ്സുചെയ്തു് ഇന്ദ്രനെ തോല്പിച്ചു് മൂന്

## 6a. Grounded explanation for each sentence (reason-then-ground, the less-bad variant from section 5)

In [11]:
SENTENCE_GROUNDED_SYSTEM = (
    "You are a Malayalam dictionary assistant. A user asked about this sentence/phrase. "
    "Below are the RAW dictionary entries found for each content word (exact text from "
    "the knowledge base, not paraphrased; some words may have no entry at all). "
    "First, reason about the sentence as a whole and explain what it means in plain "
    "English. Then, for each word that HAS a raw entry below, quote verbatim the exact "
    "substring from that word\'s entry that supports the meaning you gave it. For any "
    "word with no entry, or where no substring actually supports your reading, say so "
    "explicitly instead of forcing a match or using outside knowledge as if it were the "
    "dictionary\'s."
)


def format_word_entries(pairs: list[tuple[str, str]]) -> str:
    return "\n\n".join(f"Word: {w}\nRaw entries:\n{raw}" for w, raw in pairs)


sentence_results = {}
for s in TEST_SENTENCES:
    pairs = decompose(s)
    entries_text = format_word_entries(pairs) if pairs else "(no dictionary entries found for any word)"
    user = f"Sentence: {s}\n\n{entries_text}"
    answer = llm.complete(SENTENCE_GROUNDED_SYSTEM, user)
    sentence_results[s] = {"pairs": pairs, "answer": answer}
    print("=" * 70)
    print("SENTENCE:", s)
    print(answer)
    print()

SENTENCE: enthaado thanikk vende?
### Sentence Meaning
The sentence **"enthaado thanikk vende?"** translates to: **"What is needed for him?"** or **"What does he need?"**

Here is the breakdown:
*   **enthaado**: "What" (specifically asking about the nature or identity of something).
*   **thanikk**: "For him" (dative case of 'he').
*   **vende**: "Is needed" or "is required" (imperative/interrogative form of 'need').

---

### Dictionary Support

**1. Word: enthaado**
*   **Meaning used:** "What" (used to ask about the nature/identity of something).
*   **Dictionary Support:** The raw entry for **എന്ത്** defines it as a pronoun used to ask "what" or "which". Specifically, the text states:
    > "ചോദ്യാർഥം, ഒന്ന് ഇന്നതെന്നോ ഇന്നതരത്തിൽ ഉള്ളതെന്നോ അറിയാൻവേണ്ടി ചോദിക്കുമ്പോൾ പ്രയോഗിക്കുന്ന സർവനാമശബ്ദം."
    *(Translation of support: A pronoun used when asking to know whether something is this or of this kind.)*
    *Note: The form "enthaado" is the oblique/dative case of "entha" (എന്ത്),

SENTENCE: അസർബൈജാനിലെ കുർ-അറാസ് താഴ്ന്ന പ്രദേശങ്ങളിൽ സ്ഥിതിചെയ്യുന്ന ഒരു വലിയ ലവണജല തടാകമാണ് അഗ്ഗോൾ തടാകം
### Sentence Meaning
The sentence translates to: **"Lake Aggol is a large saltwater lake located in the low-lying areas of Kur-Aras in Azerbaijan."**

It describes the geographical location (low-lying areas of Kur-Aras, Azerbaijan), the nature of the body of water (large, saltwater), and identifies it as Lake Aggol.

---

### Word-by-Word Analysis & Dictionary Support

**1. താഴ്ന്ന (Thazhnnna - Low/Lying)**
*   **Meaning in context:** "Low-lying" or "situated at a lower elevation."
*   **Dictionary Support:** The entry for **താഴുക** includes the definition:
    > "ആഴം കൂടുക, കുഴിയുക (സമീപപ്രദേശത്തെക്കാൾ ഉയരം കുറയുക)"
    *(Meaning: To deepen, to sink (to have less height than the surrounding area))*
    *Note: The word in the sentence is the passive participle/adjective form derived from this root.*

**2. സ്ഥിതിചെയ്യുന്ന (Sthithicheyyunna - Situated/Located)**
*   **Meaning in cont

SENTENCE: ഹിന്ദു പുരാണങ്ങളിലെ അസുര രാജാവായ മഹാബലിയുടെ ഭാര്യയാണ് വിന്ധ്യാവലി
### Sentence Meaning
The sentence translates to: **"Vindhya vali is the wife of Mahabali, the Asura king from the Hindu Puranas."**

It identifies Vindhya vali as the spouse of Mahabali, a famous demon king (Asura) featured in Hindu mythology (Puranas).

---

### Word-by-Word Analysis & Dictionary Support

**1. ഹിന്ദു (Hindu)**
*   **Meaning in context:** Refers to the religion or tradition of Hinduism.
*   **Dictionary Support:**
    > "ഭാരതത്തിലെ പൗരാണികമതം, ഹിന്ദുമതം"
    *(This substring explicitly defines it as the ancient religion of Bharat/Hinduism.)*

**2. പുരാണങ്ങളിലെ (In the Puranas)**
*   **Meaning in context:** Indicates that the story belongs to the ancient texts known as Puranas.
*   **Dictionary Support:**
    > "പുരാവൃത്തം, പഴങ്കഥ; സർഗം പ്രതിസർഗം വംശം മന്വന്തരം വംശാനുചരിതം എന്നിവയഞ്ചും ഉൾപ്പെടുന്ന ഗ്രന്ഥം"
    *(This defines 'Purana' as ancient stories/myths and books containing genealogies and 

In [12]:
# Verify: for each sentence, which quoted Malayalam substrings are actually
# verbatim present in that sentence\'s own raw dictionary text (not hallucinated,
# not borrowed from a different word\'s entry).
for s in TEST_SENTENCES:
    pairs = sentence_results[s]["pairs"]
    answer = sentence_results[s]["answer"]
    all_raw = "\n".join(raw for _, raw in pairs)
    quoted = re.findall(r"[\u0d00-\u0d7f,\s]{3,}", answer)
    hits = [q.strip() for q in quoted if q.strip() and q.strip() in all_raw]
    print("SENTENCE:", s)
    print("  words with a dictionary hit:", [w for w, _ in pairs])
    print("  verbatim-grounded substrings cited:", hits)
    print()

SENTENCE: enthaado thanikk vende?
  words with a dictionary hit: ['enthaado', 'thanikk', 'vende']
  verbatim-grounded substrings cited: ['എന്ത്', 'ചോദ്യാർഥം, ഒന്ന് ഇന്നതെന്നോ ഇന്നതരത്തിൽ ഉള്ളതെന്നോ അറിയാൻവേണ്ടി ചോദിക്കുമ്പോൾ പ്രയോഗിക്കുന്ന സർവനാമശബ്ദം', 'എന്ത്', 'തനിക്ക്', 'താൻ എന്ന സർവനാമത്തിൻറെ ഉദ്ദേശികാവിഭക്തിരൂപം', 'വേണ്ടൂ', 'വേണം']

SENTENCE: അസർബൈജാനിലെ കുർ-അറാസ് താഴ്ന്ന പ്രദേശങ്ങളിൽ സ്ഥിതിചെയ്യുന്ന ഒരു വലിയ ലവണജല തടാകമാണ് അഗ്ഗോൾ തടാകം
  words with a dictionary hit: ['താഴ്ന്ന', 'സ്ഥിതിചെയ്യുന്ന', 'ഒരു', 'വലിയ', 'ലവണജല', 'തടാകമാണ്', 'തടാകം']
  verbatim-grounded substrings cited: ['താഴുക', 'ആഴം കൂടുക, കുഴിയുക', 'സമീപപ്രദേശത്തെക്കാൾ ഉയരം കുറയുക', 'സ്ഥിതി', 'നിലവിലുള്ള അവസ്ഥ', 'അവസ്ഥ', 'ഒരു', 'ഒരു', 'പലതല്ലാത്ത, ഏകമായ, ഒറ്റയായ', 'വലിയ', 'വലിയ', 'വലുതായ, ചെറുതല്ലാത്ത']

SENTENCE: ഹിന്ദു പുരാണങ്ങളിലെ അസുര രാജാവായ മഹാബലിയുടെ ഭാര്യയാണ് വിന്ധ്യാവലി
  words with a dictionary hit: ['ഹിന്ദു', 'പുരാണങ്ങളിലെ', 'അസുര', 'രാജാവായ', 'മഹാബലിയുടെ', 'ഭാര്യയാണ്', 'വിന്ധ്യാവലി']
  verbatim-grounded sub

## Section 6 conclusion — mostly positive, one genuinely good signal

Across 3 real sentences / 17 decomposed words, the reason-then-ground prompt:

- **Correctly grounded nearly every word** in a real, verbatim substring from
  its own raw dictionary entry (verified programmatically, not just by eye) —
  including fairly obscure grammatical words (`തനിക്ക്` → dative-case-of-"താൻ"
  entry, `ഒരു` → "single, solitary" entry) and a domain-specific compound
  (`തടാകം`/lake entries for the Azerbaijan sentence).
- **Correctly refused to force a match** for `അസുര` ("asura/demon" in the
  mythology sentence) — the dictionary's only real entry for that headword is
  an unrelated classical sense ("devoid of breath/life"), and the model
  explicitly said no substring supports the mythological reading, rather than
  inventing a citation. This is exactly the honest "say so instead of forcing
  a match" behavior the earlier prompt asked for, now seen working on a case
  that actually needed it.
- One minor rough edge: it occasionally lemmatizes loosely in its own head
  before quoting (e.g. explaining `താഴ്ന്ന` via the headword `താഴുക`'s
  "to sink/have less height" sense, correctly, but calling it a "passive
  participle" derivation rather than citing mlmorph) — harmless here since
  the quote itself is still correct and real.

**This contradicts the pessimistic tone of sections 4-5** (which both failed
on the single ഇരിക്കുക/"to sit" case). Taken together: grounding is not
*fundamentally* unreliable — it worked on 16 of 17 words here, and the one
non-match was the *correct* call, not a failure. ഇരിക്കുക specifically seems
to be a harder case (many near-synonymous verb senses competing, and the
correct "to sit" entry sits mid-list behind more common senses like "to
live") rather than evidence that reason-then-ground grounding doesn't work
in general.

**Practical takeaway**: the reason-then-ground grounding prompt from section
5 looks worth carrying into the real `_DECOMPOSITION_SYNTHESIS_SYSTEM` as a
prompt change — require quoting the real substring after reasoning, same as
tested here. It won't fix every hard case (ഇരിക്കുക may still need the
cleanup-cache idea, or more senses surfaced per word, or self-consistency
voting) but it clearly raises grounding fidelity on the common case without
extra infrastructure. Worth running a larger, more systematic sample before
committing to the prompt change — this is still just 4 sentences total across
this notebook (1 Manglish phrase + 3 new ones), not a proper eval set.